# BuzzSpot - YOLO26m train+val final-fit rare oversampling


## 1. Install


In [1]:
!pip install -q ultralytics pycocotools tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.1 MB/s eta 0:00:00


## 2. Mount Drive and locate dataset files


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import glob
from pathlib import Path

zip_hits = glob.glob("/content/drive/MyDrive/**/BuzzSet_challenge.zip", recursive=True)
assert zip_hits, "BuzzSet_challenge.zip not found under MyDrive"
ZIP_PATH = zip_hits[0]

tar_hits = glob.glob("/content/drive/MyDrive/**/19557529600-BuzzSet_challenge_testphase.tar", recursive=True)
if not tar_hits:
    tar_hits = glob.glob("/content/drive/MyDrive/**/*BuzzSet_challenge_testphase*.tar*", recursive=True)
assert tar_hits, "test-phase tar file not found under MyDrive"
TAR_PATH = tar_hits[0]

print("old train/valid zip:", ZIP_PATH)
print("new test-phase tar:", TAR_PATH)


Mounted at /content/drive
old train/valid zip: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/BuzzSet_challenge.zip
new test-phase tar: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/19557529600-BuzzSet_challenge_testphase.tar


## 3. Config


In [3]:
from pathlib import Path
import shutil

# Final-fit train+val run:
# - True: train YOLO26m on train + valid annotated keyframes, then save last.pt to Drive
# - False: skip training and restore saved final-fit checkpoint from Drive
ENABLE_TRAINING = True

# This is a final-fit Model A run.
# It keeps the same full-frame YOLO26m rare-class oversampling setup,
# but adds the official valid annotations into the training set.
MODEL_NAME = "yolo26m.pt"
MODEL_TAG = "yolo26m_32ep_rare_os_cm5_trainval_finalfit"
RUN_NAME = "buzz_yolo26m_32ep_rare_os_cm5_trainval_finalfit"

EPOCHS = 32
PATIENCE = 8
SAVE_PERIOD = 5

IMGSZ = 1536
BATCH = 2  # if this OOMs, use BATCH = 1

MOSAIC = 1.0
CLOSE_MOSAIC = 5

# This run trains on all official validation labels, so validation AP is no longer honest.
# Keep this False for the real final-fit run. We will submit/check the resulting checkpoint.
RUN_VALIDATION_DURING_TRAINING = False
RUN_SEEN_VAL_SANITY_CHECK = False

# Use last.pt for final submission/inference because we are not selecting by a clean validation set.
SUBMISSION_WEIGHTS_KIND = "last"  # "last" for final-fit; "best" only if you deliberately enable validation

# Default full-frame inference settings.
# CONF=0.001 matches the low-threshold A-only / ensemble sweep behavior.
CONF = 0.001
IOU = 0.60
MAX_DET = 100
AUGMENT = False

LOCAL_DATA_DIR = Path("/content/buzz_trainval_finalfit")
LOCAL_WEIGHTS = Path("/content/final_model_a_trainval.pt")

PROJECT_DIR = Path("/content/drive/MyDrive/BuzzSpot")
WEIGHTS_DIR = PROJECT_DIR / "weights"
DRIVE_RUNS_DIR = PROJECT_DIR / "runs_oversampling"
SUBMISSIONS_DIR = PROJECT_DIR / "submissions"

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_BEST_WEIGHTS = WEIGHTS_DIR / f"{MODEL_TAG}_best.pt"
DRIVE_LAST_WEIGHTS = WEIGHTS_DIR / f"{MODEL_TAG}_last.pt"
DRIVE_RESULTS_CSV = WEIGHTS_DIR / f"{MODEL_TAG}_results.csv"

CONF_TAG = f"conf{int(CONF * 1000):03d}"
LOCAL_PRED_PATH = Path("/content/predictions.json")
LOCAL_ZIP_OUT = Path(f"/content/submission_{MODEL_TAG}_{CONF_TAG}.zip")
DRIVE_PRED_PATH = SUBMISSIONS_DIR / f"predictions_{MODEL_TAG}_{CONF_TAG}.json"
DRIVE_ZIP_OUT = SUBMISSIONS_DIR / f"submission_{MODEL_TAG}_{CONF_TAG}.zip"

CLASS_NAMES = ["bee", "bumblebee", "hoverfly", "moth"]
CATEGORY_NAME_TO_ID = {name: i + 1 for i, name in enumerate(CLASS_NAMES)}

# YOLO class ids: 0 bee, 1 bumblebee, 2 hoverfly, 3 moth
# Same full-frame image-level oversampling as the best Model A run.
CLASS_MULTIPLIER = {
    0: 1,  # bee
    1: 4,  # bumblebee
    2: 2,  # hoverfly
    3: 5,  # moth
}


def selected_drive_weights_path():
    if SUBMISSION_WEIGHTS_KIND == "last":
        return DRIVE_LAST_WEIGHTS
    if SUBMISSION_WEIGHTS_KIND == "best":
        return DRIVE_BEST_WEIGHTS
    raise ValueError(f"Bad SUBMISSION_WEIGHTS_KIND: {SUBMISSION_WEIGHTS_KIND}")


def restore_submission_weights_to_local():
    """
    Restores the selected Drive checkpoint into LOCAL_WEIGHTS.
    For this final-fit train+val run, this should normally be last.pt.
    """
    if SUBMISSION_WEIGHTS_KIND == "last":
        candidates = [
            DRIVE_LAST_WEIGHTS,
            DRIVE_RUNS_DIR / RUN_NAME / "weights" / "last.pt",
        ]
    elif SUBMISSION_WEIGHTS_KIND == "best":
        candidates = [
            DRIVE_BEST_WEIGHTS,
            DRIVE_RUNS_DIR / RUN_NAME / "weights" / "best.pt",
        ]
    else:
        raise ValueError(f"Bad SUBMISSION_WEIGHTS_KIND: {SUBMISSION_WEIGHTS_KIND}")

    for p in candidates:
        if p.exists():
            shutil.copy(p, LOCAL_WEIGHTS)
            print(f"Restored {SUBMISSION_WEIGHTS_KIND}.pt from Drive:", p)
            print("Local weights:", LOCAL_WEIGHTS)
            return LOCAL_WEIGHTS

    if LOCAL_WEIGHTS.exists():
        print("WARNING: Drive backup not found, but local weights exist:", LOCAL_WEIGHTS)
        return LOCAL_WEIGHTS

    raise FileNotFoundError(
        f"No saved {SUBMISSION_WEIGHTS_KIND}.pt found. Set ENABLE_TRAINING = True and run training first."
    )


def backup_training_outputs_to_drive():
    """
    Copies best.pt, last.pt, and results.csv from the Drive run folder
    into stable Drive backup paths. The selected checkpoint is copied to LOCAL_WEIGHTS.
    """
    run_dir = DRIVE_RUNS_DIR / RUN_NAME
    best_path = run_dir / "weights" / "best.pt"
    last_path = run_dir / "weights" / "last.pt"
    results_csv = run_dir / "results.csv"

    if best_path.exists():
        shutil.copy(best_path, DRIVE_BEST_WEIGHTS)
        print("Backed up best weights to Drive:", DRIVE_BEST_WEIGHTS)
    else:
        print("No best.pt found. This is expected if validation was disabled.")

    if last_path.exists():
        shutil.copy(last_path, DRIVE_LAST_WEIGHTS)
        print("Backed up last weights to Drive:", DRIVE_LAST_WEIGHTS)
    else:
        raise FileNotFoundError(f"last.pt not found at {last_path}")

    if results_csv.exists():
        shutil.copy(results_csv, DRIVE_RESULTS_CSV)
        print("Backed up results.csv to Drive:", DRIVE_RESULTS_CSV)

    selected = selected_drive_weights_path()
    assert selected.exists(), f"Selected checkpoint not found: {selected}"
    shutil.copy(selected, LOCAL_WEIGHTS)
    print(f"Copied selected {SUBMISSION_WEIGHTS_KIND}.pt to local:", LOCAL_WEIGHTS)

print("ENABLE_TRAINING:", ENABLE_TRAINING)
print("model:", MODEL_NAME)
print("model tag:", MODEL_TAG)
print("epochs:", EPOCHS)
print("patience:", PATIENCE)
print("imgsz:", IMGSZ)
print("batch:", BATCH)
print("mosaic:", MOSAIC)
print("close_mosaic:", CLOSE_MOSAIC)
print("train-time validation enabled:", RUN_VALIDATION_DURING_TRAINING)
print("submission weights:", SUBMISSION_WEIGHTS_KIND)
print("run:", RUN_NAME)
print("Drive last weights:", DRIVE_LAST_WEIGHTS)
print("Drive zip output:", DRIVE_ZIP_OUT)
print("oversampling:", {CLASS_NAMES[k]: v for k, v in CLASS_MULTIPLIER.items()})

ENABLE_TRAINING: True
model: yolo26m.pt
model tag: yolo26m_32ep_rare_os_cm5_trainval_finalfit
epochs: 32
patience: 8
imgsz: 1536
batch: 2
mosaic: 1.0
close_mosaic: 5
train-time validation enabled: False
submission weights: last
run: buzz_yolo26m_32ep_rare_os_cm5_trainval_finalfit
Drive last weights: /content/drive/MyDrive/BuzzSpot/weights/yolo26m_32ep_rare_os_cm5_trainval_finalfit_last.pt
Drive zip output: /content/drive/MyDrive/BuzzSpot/submissions/submission_yolo26m_32ep_rare_os_cm5_trainval_finalfit_conf001.zip
oversampling: {'bee': 1, 'bumblebee': 4, 'hoverfly': 2, 'moth': 5}


## 4. Build YOLO dataset + train+val rare-class oversampled train list


In [4]:
import json, zipfile, tarfile, shutil
from pathlib import Path
from collections import defaultdict, Counter

# clean old local dataset
if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)

for d in [
    "images/train", "images/val", "images/test_testphase",
    "labels/train", "labels/val", "annotations"
]:
    (LOCAL_DATA_DIR / d).mkdir(parents=True, exist_ok=True)

zf = zipfile.ZipFile(ZIP_PATH)
tf = tarfile.open(TAR_PATH, "r:*")

zip_names = set(zf.namelist())
tar_members = {m.name: m for m in tf.getmembers() if m.isfile()}
tar_names = set(tar_members.keys())


def find_zip(rel):
    rel = rel.lstrip("/")
    for cand in (rel, f"BuzzSet_challenge/{rel}"):
        if cand in zip_names:
            return cand
    suffix = "/" + rel
    hits = [n for n in zip_names if n.endswith(suffix)]
    return hits[0] if hits else None


def find_tar(rel):
    rel = rel.lstrip("/")
    for cand in (rel, f"BuzzSet_challenge/{rel}", f"BuzzSet_challenge_testphase/{rel}"):
        if cand in tar_names:
            return cand
    suffix = "/" + rel
    hits = [n for n in tar_names if n.endswith(suffix)]
    return hits[0] if hits else None


def load_tar_json(fname):
    p = find_tar(f"annotations/{fname}") or find_tar(fname)
    assert p is not None, f"{fname} not found in tar"
    with tf.extractfile(tar_members[p]) as f:
        obj = json.load(f)
    out = LOCAL_DATA_DIR / "annotations" / fname
    out.write_text(json.dumps(obj))
    print("loaded current annotation:", fname, "from", p)
    return obj


def flat_name(file_name):
    return file_name.replace("/", "__")


def write_yolo_label(label_path, anns, W, H):
    lines = []
    for ann in anns:
        x, y, w, h = ann["bbox"]
        cls = int(ann["category_id"]) - 1

        # clip just in case
        x = max(0.0, min(float(x), W - 1))
        y = max(0.0, min(float(y), H - 1))
        w = max(0.0, min(float(w), W - x))
        h = max(0.0, min(float(h), H - y))

        if w < 1 or h < 1:
            continue

        xc = (x + w / 2) / W
        yc = (y + h / 2) / H
        ww = w / W
        hh = h / H

        lines.append(f"{cls} {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}")

    label_path.write_text("\n".join(lines))


def extract_annotated_split(coco, zip_img_dir, out_split):
    by_img = defaultdict(list)
    for ann in coco.get("annotations", []):
        by_img[int(ann["image_id"])].append(ann)

    images_by_id = {int(im["id"]): im for im in coco["images"]}
    image_ids = sorted(by_img.keys())

    written_images = []
    count_images = 0
    count_boxes = 0
    class_counts = Counter()

    for iid in image_ids:
        im = images_by_id[iid]
        src = find_zip(f"{zip_img_dir}/{im['file_name']}") or find_zip(im["file_name"])
        assert src is not None, f"missing image in old zip: {zip_img_dir}/{im['file_name']}"

        out_name = flat_name(im["file_name"])
        img_dst = LOCAL_DATA_DIR / "images" / out_split / out_name
        lbl_dst = LOCAL_DATA_DIR / "labels" / out_split / (Path(out_name).stem + ".txt")

        with zf.open(src) as s, open(img_dst, "wb") as d:
            shutil.copyfileobj(s, d)

        anns = by_img[iid]
        write_yolo_label(lbl_dst, anns, int(im["width"]), int(im["height"]))

        written_images.append(img_dst)
        count_images += 1
        count_boxes += len(anns)
        for ann in anns:
            class_counts[int(ann["category_id"])] += 1

    print(out_split, "images:", count_images, "boxes:", count_boxes, "class counts:", dict(class_counts))
    print(out_split, "named counts:", {CLASS_NAMES[k-1]: class_counts[k] for k in range(1, 5)})
    return written_images


def extract_test_keyframes(test_coco):
    keyframe_images = [
        im for im in test_coco["images"]
        if im.get("is_keyframe") in [True, 1, "true", "True"]
    ]

    flat_to_id = {}

    for im in keyframe_images:
        src = find_tar(f"test_testphase/{im['file_name']}") or find_tar(im["file_name"])
        assert src is not None, f"missing test image in tar: test_testphase/{im['file_name']}"

        out_name = flat_name(im["file_name"])
        dst = LOCAL_DATA_DIR / "images" / "test_testphase" / out_name

        with tf.extractfile(tar_members[src]) as s, open(dst, "wb") as d:
            shutil.copyfileobj(s, d)

        flat_to_id[out_name] = int(im["id"])

    keyframe_paths = [
        str(LOCAL_DATA_DIR / "images" / "test_testphase" / flat_name(im["file_name"]))
        for im in keyframe_images
    ]

    print("test_testphase keyframes:", len(keyframe_paths))
    return keyframe_images, flat_to_id, keyframe_paths


def yolo_label_path_from_image_path(img_path: Path) -> Path:
    s = str(img_path)
    assert "/images/" in s, f"Cannot infer label path from image path: {img_path}"
    return Path(s.replace("/images/", "/labels/", 1)).with_suffix(".txt")


def classes_in_label(label_path: Path):
    classes = set()
    if not label_path.exists():
        return classes
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                classes.add(int(float(parts[0])))
    return classes


def count_instances(label_path: Path):
    counts = Counter()
    if not label_path.exists():
        return counts
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                counts[int(float(parts[0]))] += 1
    return counts


def build_oversampled_train_txt(train_image_paths):
    oversampled_lines = []
    image_multiplier_counts = Counter()
    class_image_counts = Counter()
    original_instance_counts = Counter()
    effective_instance_counts = Counter()

    for img_path in train_image_paths:
        lp = yolo_label_path_from_image_path(img_path)
        cls_set = classes_in_label(lp)
        inst_counts = count_instances(lp)

        for c in cls_set:
            class_image_counts[c] += 1
        original_instance_counts.update(inst_counts)

        mult = max([CLASS_MULTIPLIER.get(c, 1) for c in cls_set] or [1])
        image_multiplier_counts[mult] += 1

        for _ in range(mult):
            oversampled_lines.append(str(img_path.resolve()))

        for c, n in inst_counts.items():
            effective_instance_counts[c] += n * mult

    train_txt = LOCAL_DATA_DIR / "train_oversampled.txt"
    train_txt.write_text("\n".join(oversampled_lines) + "\n")

    print("\noversampling summary")
    print("original full-frame train+val images:", len(train_image_paths))
    print("total train entries:", len(oversampled_lines))
    print("full-frame train+val slowdown factor:", round(len(oversampled_lines) / len(train_image_paths), 3))
    print("image multiplier counts:", dict(sorted(image_multiplier_counts.items())))

    print("\nclass image counts before oversampling/full-frame train+val:")
    for k, name in enumerate(CLASS_NAMES):
        print(f"  {name:10s}: {class_image_counts[k]}")

    print("\ninstance counts before oversampling/full-frame train+val:")
    for k, name in enumerate(CLASS_NAMES):
        print(f"  {name:10s}: {original_instance_counts[k]}")

    print("\neffective instance counts after oversampling:")
    for k, name in enumerate(CLASS_NAMES):
        print(f"  {name:10s}: {effective_instance_counts[k]}")

    print("\nwrote:", train_txt)
    return train_txt

# Current annotations come from the new tar.
# Train/valid images come from the old zip.
# Test images/json come from the new tar.
train = load_tar_json("train.json")
valid = load_tar_json("valid.json")
test = load_tar_json("test_testphase.json")

train_image_paths = extract_annotated_split(train, "train", "train")
valid_image_paths = extract_annotated_split(valid, "valid", "val")
keyframe_images, flat_to_id, keyframe_paths = extract_test_keyframes(test)


# Final-fit training set: official train + official valid annotated keyframes.
TRAINVAL_IMAGE_PATHS = train_image_paths + valid_image_paths
print("\nfinal-fit train+val images:", len(TRAINVAL_IMAGE_PATHS))
print("  train images:", len(train_image_paths))
print("  valid images added to train:", len(valid_image_paths))
OVERSAMPLED_TRAIN_TXT = build_oversampled_train_txt(TRAINVAL_IMAGE_PATHS)

zf.close()
tf.close()

RAW_DATA_YAML = LOCAL_DATA_DIR / "data_raw.yaml"
RAW_DATA_YAML.write_text(
    f"path: {LOCAL_DATA_DIR}\n"
    "train: images/train\n"
    "val: images/val\n"
    "nc: 4\n"
    "names: ['bee', 'bumblebee', 'hoverfly', 'moth']\n"
)

DATA_YAML = LOCAL_DATA_DIR / "data_trainval_oversampled.yaml"
DATA_YAML.write_text(
    f"path: {LOCAL_DATA_DIR}\n"
    "train: train_oversampled.txt\n"
    "val: images/val\n"  # seen during training; not an honest validation set for final-fit

    "nc: 4\n"
    "names: ['bee', 'bumblebee', 'hoverfly', 'moth']\n"
)

print("\nraw data yaml:")
print(RAW_DATA_YAML.read_text())

print("train+val oversampled data yaml used for final-fit training:")
print(DATA_YAML.read_text())


loaded current annotation: train.json from BuzzSpot_testphase/annotations/train.json
loaded current annotation: valid.json from BuzzSpot_testphase/annotations/valid.json
loaded current annotation: test_testphase.json from BuzzSpot_testphase/annotations/test_testphase.json
train images: 5275 boxes: 10884 class counts: {1: 8677, 3: 1753, 2: 237, 4: 217}
train named counts: {'bee': 8677, 'bumblebee': 237, 'hoverfly': 1753, 'moth': 217}
val images: 932 boxes: 1116 class counts: {1: 934, 2: 30, 3: 95, 4: 57}
val named counts: {'bee': 934, 'bumblebee': 30, 'hoverfly': 95, 'moth': 57}
test_testphase keyframes: 4763

final-fit train+val images: 6207
  train images: 5275
  valid images added to train: 932

oversampling summary
original full-frame train+val images: 6207
total train entries: 9253
full-frame train+val slowdown factor: 1.491
image multiplier counts: {1: 4418, 2: 1286, 4: 252, 5: 251}

class image counts before oversampling/full-frame train+val:
  bee       : 5703
  bumblebee : 263


## 5. Train YOLO26m final-fit on train + valid


In [5]:
from ultralytics import YOLO
import torch, gc

if ENABLE_TRAINING:
    print("ENABLE_TRAINING=True -> training will run.")
    print("Final-fit run: training on official train + official valid annotations.")
    print("Validation during training:", RUN_VALIDATION_DURING_TRAINING)
    print("Checkpoint to use for submission:", SUBMISSION_WEIGHTS_KIND)

    model = YOLO(MODEL_NAME)

    train_kwargs = dict(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        save_period=SAVE_PERIOD,
        project=str(DRIVE_RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        seed=0,
        deterministic=True,
        cache="disk",
        workers=4,
        device=0,
        optimizer="auto",
        mosaic=MOSAIC,
        close_mosaic=CLOSE_MOSAIC,
        val=RUN_VALIDATION_DURING_TRAINING,
    )

    print("training args:", train_kwargs)
    model.train(**train_kwargs)

    backup_training_outputs_to_drive()

    gc.collect()
    torch.cuda.empty_cache()

else:
    print("ENABLE_TRAINING=False -> skipping training.")
    restore_submission_weights_to_local()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ENABLE_TRAINING=True -> training will run.
Final-fit run: training on official train + official valid annotations.
Validation during training: False
Checkpoint to use for submission: last
training args: {'data': '/content/buzz_trainval_finalfit/data_trainval_oversampled.yaml', 'epochs': 32, 'imgsz': 1536, 'batch': 2, 'patience': 8, 'save_period': 5, 'project': '/content/drive/MyDrive/BuzzSpot/runs_oversampling', 'name': 'buzz_yolo26m_32ep_rare_os_cm5_trainval_finalfit', 'exist_ok': True, 'seed': 0, 'deterministic': True, 'cache': 'disk', 'workers': 4, 'device': 0, 'optimizer': 'auto', 'mosaic': 1.0, 'close_mosaic': 5, 'val': False}
Ultralytics 8.4.84 🚀 Python-3.12.13 torch-2.11.0+

## 6. Optional seen-val sanity check

This run trains on official valid annotations, so this AP is not an honest model-selection metric.


In [6]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

restore_submission_weights_to_local()

if not RUN_SEEN_VAL_SANITY_CHECK:
    print("Skipping seen-val AP sanity check.")
    print("This train+val run uses official validation labels for training, so local val AP is not honest.")
    print("Selected weights:", LOCAL_WEIGHTS)
else:
    model = YOLO(str(LOCAL_WEIGHTS))
    print("classes:", model.names)

    print("\nUsing existing notebook config:")
    print("CONF:", CONF)
    print("IOU:", IOU)
    print("MAX_DET:", MAX_DET)
    print("IMGSZ:", IMGSZ)

    metrics = model.val(
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        batch=1,
        split="val",
        conf=CONF,
        iou=IOU,
        max_det=MAX_DET,
        agnostic_nms=False,
        plots=False,
        verbose=False,
    )

    row = {
        "conf": CONF,
        "iou": IOU,
        "max_det": MAX_DET,
        "map50_95_seen_val": float(metrics.box.map),
        "map50_seen_val": float(metrics.box.map50),
        "map75_seen_val": float(metrics.box.map75),
    }

    for cls_idx, cls_name in model.names.items():
        row[f"seen_val_ap_{cls_name}"] = float(metrics.box.maps[cls_idx])

    df = pd.DataFrame([row])
    display(df)

Restored last.pt from Drive: /content/drive/MyDrive/BuzzSpot/weights/yolo26m_32ep_rare_os_cm5_trainval_finalfit_last.pt
Local weights: /content/final_model_a_trainval.pt
Skipping seen-val AP sanity check.
This train+val run uses official validation labels for training, so local val AP is not honest.
Selected weights: /content/final_model_a_trainval.pt


In [7]:
# Optional visual inspect on seen validation images.
# This is only for qualitative sanity checking, not model selection.
RUN_VISUAL_SANITY_CHECK = False

if not RUN_VISUAL_SANITY_CHECK:
    print("Skipping visual sanity check on seen validation images.")
else:
    # Visual inspect: GT vs full-frame predictions on 100 random validation images
    # Uses existing CONF / IOU / MAX_DET / IMGSZ exactly as defined above.
    # No SAHI.

    import random, gc, torch, yaml
    from pathlib import Path
    from PIL import Image, ImageDraw, ImageFont
    import matplotlib.pyplot as plt

    # -----------------------
    # Config
    # -----------------------
    NUM_RANDOM_IMAGES = 100
    RANDOM_SEED = 7

    ZOOM_TO_BOXES = True
    CROP_MARGIN = 180
    MIN_CROP_SIZE = 500

    LINE_WIDTH = 5
    FONT_SIZE = 24

    CLASS_NAMES = ["bee", "bumblebee", "hoverfly", "moth"]

    GT_COLOR = (0, 255, 0)       # green
    PRED_COLOR = (255, 0, 0)     # red

    try:
        FONT = ImageFont.truetype("DejaVuSans-Bold.ttf", FONT_SIZE)
    except:
        FONT = None


    # -----------------------
    # Resolve validation image paths
    # -----------------------
    def resolve_yolo_path(p, yaml_path):
        p = Path(p)
        if p.is_absolute():
            return p
        return Path(yaml_path).parent / p


    def get_val_image_paths_from_yaml(data_yaml):
        data_yaml = Path(data_yaml)

        with open(data_yaml, "r") as f:
            cfg = yaml.safe_load(f)

        base = cfg.get("path", "")
        if base:
            base = Path(base)
            if not base.is_absolute():
                base = data_yaml.parent / base
        else:
            base = data_yaml.parent

        val_entry = cfg["val"]
        val_path = Path(val_entry)

        if not val_path.is_absolute():
            val_path = base / val_path

        if val_path.is_file() and val_path.suffix == ".txt":
            lines = val_path.read_text().strip().splitlines()
            paths = []
            for line in lines:
                p = Path(line.strip())
                if not p.is_absolute():
                    p = val_path.parent / p
                paths.append(p)
            return paths

        exts = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
        paths = []
        for ext in exts:
            paths.extend(sorted(val_path.glob(ext)))

        return paths


    try:
        val_paths = list(valid_image_paths)
    except NameError:
        val_paths = get_val_image_paths_from_yaml(DATA_YAML)

    val_paths = [Path(p) for p in val_paths]
    print("Validation images found:", len(val_paths))


    # -----------------------
    # Label lookup
    # -----------------------
    def label_path_for_img(img_path):
        img_path = Path(img_path)

        # Prefer explicit val_label_dir if notebook has it
        if "val_label_dir" in globals():
            p = Path(val_label_dir) / f"{img_path.stem}.txt"
            if p.exists():
                return p

        # Standard YOLO layout: .../images/valid/x.jpg -> .../labels/valid/x.txt
        parts = list(img_path.parts)
        if "images" in parts:
            idx = parts.index("images")
            parts[idx] = "labels"
            p = Path(*parts).with_suffix(".txt")
            if p.exists():
                return p

        # Common fallback: sibling labels dir
        candidates = [
            img_path.parent.parent / "labels" / img_path.parent.name / f"{img_path.stem}.txt",
            img_path.parent.parent / "labels" / f"{img_path.stem}.txt",
            img_path.parent / f"{img_path.stem}.txt",
        ]

        for p in candidates:
            if p.exists():
                return p

        return None


    def read_gt_boxes_from_yolo_label(img_path):
        img = Image.open(img_path)
        W, H = img.size

        label_path = label_path_for_img(img_path)
        boxes = []

        if label_path is None or not label_path.exists():
            return boxes

        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue

                cls = int(float(parts[0]))
                xc, yc, bw, bh = map(float, parts[1:5])

                x1 = (xc - bw / 2) * W
                y1 = (yc - bh / 2) * H
                x2 = (xc + bw / 2) * W
                y2 = (yc + bh / 2) * H

                boxes.append({
                    "category_id": cls + 1,
                    "bbox_xyxy": [x1, y1, x2, y2],
                    "score": None,
                })

        return boxes


    # -----------------------
    # Prediction
    # -----------------------
    def predict_one_image_full_frame(img_path):
        preds = []

        with torch.inference_mode():
            r = model.predict(
                source=str(img_path),
                imgsz=IMGSZ,
                conf=CONF,
                iou=IOU,
                max_det=MAX_DET,
                batch=1,
                augment=AUGMENT if "AUGMENT" in globals() else False,
                agnostic_nms=False,
                verbose=False,
            )[0]

        if r.boxes is not None and len(r.boxes) > 0:
            for b in r.boxes:
                x1, y1, x2, y2 = b.xyxy[0].tolist()

                # clip to actual image size, not hardcoded 1920x1080
                img = Image.open(img_path)
                W, H = img.size

                x1 = max(0.0, min(float(x1), W - 1))
                y1 = max(0.0, min(float(y1), H - 1))
                x2 = max(0.0, min(float(x2), W))
                y2 = max(0.0, min(float(y2), H))

                if x2 > x1 and y2 > y1:
                    preds.append({
                        "category_id": int(b.cls[0]) + 1,
                        "bbox_xyxy": [x1, y1, x2, y2],
                        "score": float(b.conf[0]),
                    })

        del r
        return preds


    # -----------------------
    # Drawing
    # -----------------------
    def get_crop_box(img, gt_boxes, pred_boxes):
        W, H = img.size
        all_boxes = gt_boxes + pred_boxes

        if not ZOOM_TO_BOXES or len(all_boxes) == 0:
            return (0, 0, W, H)

        xs1 = [b["bbox_xyxy"][0] for b in all_boxes]
        ys1 = [b["bbox_xyxy"][1] for b in all_boxes]
        xs2 = [b["bbox_xyxy"][2] for b in all_boxes]
        ys2 = [b["bbox_xyxy"][3] for b in all_boxes]

        x1 = max(0, int(min(xs1) - CROP_MARGIN))
        y1 = max(0, int(min(ys1) - CROP_MARGIN))
        x2 = min(W, int(max(xs2) + CROP_MARGIN))
        y2 = min(H, int(max(ys2) + CROP_MARGIN))

        crop_w = x2 - x1
        crop_h = y2 - y1

        if crop_w < MIN_CROP_SIZE:
            extra = (MIN_CROP_SIZE - crop_w) // 2
            x1 = max(0, x1 - extra)
            x2 = min(W, x2 + extra)

        if crop_h < MIN_CROP_SIZE:
            extra = (MIN_CROP_SIZE - crop_h) // 2
            y1 = max(0, y1 - extra)
            y2 = min(H, y2 + extra)

        return (x1, y1, x2, y2)


    def draw_boxes(img, boxes, crop_box, color, is_prediction=False):
        out = img.crop(crop_box).convert("RGB")
        draw = ImageDraw.Draw(out)

        ox, oy, _, _ = crop_box

        for b in boxes:
            x1, y1, x2, y2 = b["bbox_xyxy"]
            x1 -= ox
            x2 -= ox
            y1 -= oy
            y2 -= oy

            cls_id = int(b["category_id"])
            cls_name = CLASS_NAMES[cls_id - 1]

            if is_prediction:
                label = f"{cls_name} {b['score']:.3f}"
            else:
                label = f"GT {cls_name}"

            draw.rectangle([x1, y1, x2, y2], outline=color, width=LINE_WIDTH)

            text_x = max(0, int(x1))
            text_y = max(0, int(y1) - FONT_SIZE - 8)

            if FONT is not None:
                tb = draw.textbbox((text_x, text_y), label, font=FONT)
                draw.rectangle(
                    [tb[0] - 4, tb[1] - 4, tb[2] + 4, tb[3] + 4],
                    fill=color,
                )
                draw.text((text_x, text_y), label, fill=(0, 0, 0), font=FONT)
            else:
                draw.rectangle([text_x, text_y, text_x + 220, text_y + 28], fill=color)
                draw.text((text_x + 3, text_y + 3), label, fill=(0, 0, 0))

        return out


    # -----------------------
    # Sample and show
    # -----------------------
    rng = random.Random(RANDOM_SEED)
    sample_paths = rng.sample(val_paths, k=min(NUM_RANDOM_IMAGES, len(val_paths)))

    print("\nShowing", len(sample_paths), "random val images")
    print("Mode: full_frame")
    print("CONF:", CONF)
    print("IOU:", IOU)
    print("MAX_DET:", MAX_DET)
    print("IMGSZ:", IMGSZ)
    print("AUGMENT:", AUGMENT if "AUGMENT" in globals() else False)
    print("ZOOM_TO_BOXES:", ZOOM_TO_BOXES)

    for idx, img_path in enumerate(sample_paths, start=1):
        img_path = Path(img_path)

        gt_boxes = read_gt_boxes_from_yolo_label(img_path)
        pred_boxes = predict_one_image_full_frame(img_path)

        img = Image.open(img_path).convert("RGB")
        crop_box = get_crop_box(img, gt_boxes, pred_boxes)

        gt_view = draw_boxes(
            img,
            gt_boxes,
            crop_box,
            color=GT_COLOR,
            is_prediction=False,
        )

        pred_view = draw_boxes(
            img,
            pred_boxes,
            crop_box,
            color=PRED_COLOR,
            is_prediction=True,
        )

        print(
            f"\n[{idx}/{len(sample_paths)}] {img_path.name} | "
            f"GT={len(gt_boxes)} | Pred={len(pred_boxes)}"
        )

        fig, axes = plt.subplots(1, 2, figsize=(24, 9), dpi=120)

        axes[0].imshow(gt_view)
        axes[0].set_title(f"Ground Truth | {img_path.name}", fontsize=16)
        axes[0].axis("off")

        axes[1].imshow(pred_view)
        axes[1].set_title("Prediction | full_frame", fontsize=16)
        axes[1].axis("off")

        plt.tight_layout()
        plt.show()

        if idx % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

Skipping visual sanity check on seen validation images.


## 7. Full-frame inference on test_testphase keyframes


In [8]:
import json, zipfile, gc, torch, shutil
from pathlib import Path
from ultralytics import YOLO

restore_submission_weights_to_local()

model = YOLO(str(LOCAL_WEIGHTS))

try:
    del results
except:
    pass

gc.collect()
torch.cuda.empty_cache()

W, H = 1920, 1080
preds = []

for i, img_path in enumerate(keyframe_paths):
    with torch.inference_mode():
        r = model.predict(
            source=str(img_path),
            imgsz=IMGSZ,
            conf=CONF,
            iou=IOU,
            max_det=MAX_DET,
            batch=1,
            augment=AUGMENT,
            verbose=False
        )[0]

    fname = Path(r.path).name
    iid = flat_to_id.get(fname)

    if iid is not None:
        for b in r.boxes:
            x1, y1, x2, y2 = b.xyxy[0].tolist()

            x1 = max(0.0, min(x1, W - 1))
            y1 = max(0.0, min(y1, H - 1))
            x2 = max(0.0, min(x2, W))
            y2 = max(0.0, min(y2, H))

            w = x2 - x1
            h = y2 - y1

            if w >= 1 and h >= 1:
                preds.append({
                    "image_id": int(iid),
                    "category_id": int(b.cls[0]) + 1,
                    "bbox": [round(x1, 2), round(y1, 2), round(w, 2), round(h, 2)],
                    "score": round(float(b.conf[0]), 5),
                })

    del r

    if i % 100 == 0:
        print(i, "/", len(keyframe_paths), "preds:", len(preds))
        gc.collect()
        torch.cuda.empty_cache()

with open(LOCAL_PRED_PATH, "w") as f:
    json.dump(preds, f)

with zipfile.ZipFile(LOCAL_ZIP_OUT, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(LOCAL_PRED_PATH, arcname="predictions.json")

shutil.copy(LOCAL_PRED_PATH, DRIVE_PRED_PATH)
shutil.copy(LOCAL_ZIP_OUT, DRIVE_ZIP_OUT)

print("detections:", len(preds))
print("local predictions:", LOCAL_PRED_PATH)
print("local zip:", LOCAL_ZIP_OUT)
print("Drive predictions:", DRIVE_PRED_PATH)
print("Drive zip:", DRIVE_ZIP_OUT)


Restored last.pt from Drive: /content/drive/MyDrive/BuzzSpot/weights/yolo26m_32ep_rare_os_cm5_trainval_finalfit_last.pt
Local weights: /content/final_model_a_trainval.pt
0 / 4763 preds: 11
100 / 4763 preds: 713
200 / 4763 preds: 984
300 / 4763 preds: 1220
400 / 4763 preds: 1424
500 / 4763 preds: 1728
600 / 4763 preds: 1976
700 / 4763 preds: 2216
800 / 4763 preds: 2452
900 / 4763 preds: 2793
1000 / 4763 preds: 3310
1100 / 4763 preds: 4085
1200 / 4763 preds: 4683
1300 / 4763 preds: 5242
1400 / 4763 preds: 5990
1500 / 4763 preds: 6689
1600 / 4763 preds: 7174
1700 / 4763 preds: 7731
1800 / 4763 preds: 8411
1900 / 4763 preds: 9061
2000 / 4763 preds: 9604
2100 / 4763 preds: 10185
2200 / 4763 preds: 11292
2300 / 4763 preds: 12496
2400 / 4763 preds: 13532
2500 / 4763 preds: 15461
2600 / 4763 preds: 17965
2700 / 4763 preds: 18957
2800 / 4763 preds: 19954
2900 / 4763 preds: 20520
3000 / 4763 preds: 21453
3100 / 4763 preds: 22270
3200 / 4763 preds: 22906
3300 / 4763 preds: 23475
3400 / 4763 preds

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 8. Validate submission zip before upload


In [10]:
import json, zipfile
from pathlib import Path
from collections import Counter

p = json.load(open(LOCAL_PRED_PATH))
ids = {int(d["image_id"]) for d in p}
keyframe_ids = {int(im["id"]) for im in keyframe_images}

print("detections:", len(p))
print("distinct predicted images:", len(ids))
print("total keyframe images:", len(keyframe_ids))
print("predictions outside keyframes:", len(ids - keyframe_ids))
print("keyframes with no detections:", len(keyframe_ids - ids))
print("categories:", sorted({int(d["category_id"]) for d in p}))
print("class counts:", dict(Counter(int(d["category_id"]) for d in p)))

print("degenerate:", sum(1 for d in p if d["bbox"][2] < 1 or d["bbox"][3] < 1))
print("out of bounds:", sum(
    1 for d in p
    if d["bbox"][0] < -0.5
    or d["bbox"][1] < -0.5
    or d["bbox"][0] + d["bbox"][2] > 1920.5
    or d["bbox"][1] + d["bbox"][3] > 1080.5
))

with zipfile.ZipFile(LOCAL_ZIP_OUT) as z:
    contents = z.namelist()
    print("zip contents:", contents)

assert len(ids - keyframe_ids) == 0, "Submission has predictions outside keyframes"
assert sorted({int(d["category_id"]) for d in p}) and set(int(d["category_id"]) for d in p).issubset({1,2,3,4}), "Bad category ids"
assert all(d["bbox"][2] >= 1 and d["bbox"][3] >= 1 for d in p), "Degenerate boxes exist"
assert contents == ["predictions.json"], "Zip must contain exactly predictions.json"

print("\nUpload this zip:")
print(DRIVE_ZIP_OUT)


detections: 45883
distinct predicted images: 4721
total keyframe images: 4763
predictions outside keyframes: 0
keyframes with no detections: 42
categories: [1, 2, 3, 4]
class counts: {1: 26205, 3: 17653, 2: 1223, 4: 802}
degenerate: 0
out of bounds: 0
zip contents: ['predictions.json']

Upload this zip:
/content/drive/MyDrive/BuzzSpot/submissions/submission_yolo26m_32ep_rare_os_cm5_trainval_finalfit_conf001.zip
